# 04 Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
input_path = '../01_Dataset/Clean_Data/NovaMart_Retail_Cleaned.csv'
output_path = '../01_Dataset/Clean_Data/NovaMart_Retail_Engineered.csv'

df = pd.read_csv(input_path)
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

In [3]:
# Time features
if 'Order Date' in df.columns:
    df['Year'] = df['Order Date'].dt.year
    df['Quarter'] = 'Q' + df['Order Date'].dt.quarter.astype('Int64').astype(str)
    df['Month Name'] = df['Order Date'].dt.month_name()
    df['Month Number'] = df['Order Date'].dt.month
    df['Day Name'] = df['Order Date'].dt.day_name()

In [4]:
# Operational and profitability features
if set(['Order Date', 'Ship Date']).issubset(df.columns):
    df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

if set(['Profit', 'Sales']).issubset(df.columns):
    df['Profit Margin %'] = np.where(df['Sales'] != 0, (df['Profit'] / df['Sales']) * 100, np.nan)

In [5]:
# Banding features
if 'Discount' in df.columns:
    df['Discount Band'] = pd.cut(
        df['Discount'],
        bins=[-np.inf, 0, 0.1, 0.2, np.inf],
        labels=['No Discount', 'Low', 'Medium', 'High']
    )

if 'Sales' in df.columns:
    sales_q = df['Sales'].quantile([0.33, 0.66]).values
    df['Sales Band'] = pd.cut(
        df['Sales'],
        bins=[-np.inf, sales_q[0], sales_q[1], np.inf],
        labels=['Low', 'Medium', 'High']
    )

if 'Quantity' in df.columns:
    df['Order Size'] = pd.cut(
        df['Quantity'],
        bins=[-np.inf, 2, 5, np.inf],
        labels=['Small', 'Medium', 'Large']
    )

In [ ]:
df.to_csv(output_path, index=False)
print('Engineered dataset saved to:', output_path)
df.head()